In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report
)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
from google.colab import files

uploaded = files.upload()

Saving R1_parameters_dataset.csv to R1_parameters_dataset.csv


In [5]:
df = pd.read_csv("R1_parameters_dataset.csv")

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Dataset loaded successfully!
Rows: 48
Columns: 13


,rule_id,score,decision,parameter_1,parameter_2,parameter_3,parameter_4,parameter_5,parameter_6,parameter_7,parameter_8,parameter_9,parameter_10
0,R1 · 48,0.9345,APPROVE,75.0,100,10,0,A,20,20,2.5,300,1.0
1,R1 · 47,0.9274,APPROVE,75.0,100,10,0,A,20,20,2.5,300,0.9
2,R1 · 46,0.8411,APPROVE,75.0,100,10,0,A,20,20,2.5,300,0.5
3,R1 · 45,0.6925,APPROVE,75.0,100,10,0,A,20,20,2.5,300,0.1
4,R1 · 44,0.8411,APPROVE,75.0,100,10,0,A,20,20,2.5,300,0.5


In [6]:
feature_columns = [
    "parameter_1",
    "parameter_2",
    "parameter_3",
    "parameter_4",
    "parameter_5",
    "parameter_6",
    "parameter_7",
    "parameter_8",
    "parameter_9",
    "parameter_10"
]

X = df[feature_columns]

y_score = df["score"]

y_decision = df["decision"]

print("Input features:", X.shape)
print("Score target:", y_score.shape)
print("Decision target:", y_decision.shape)

Input features: (48, 10)
Score target: (48,)
Decision target: (48,)


In [7]:
categorical_features = ["parameter_5"]

numeric_features = [
    "parameter_1",
    "parameter_2",
    "parameter_3",
    "parameter_4",
    "parameter_6",
    "parameter_7",
    "parameter_8",
    "parameter_9",
    "parameter_10"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessing configured successfully!")

Preprocessing configured successfully!


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_score,
    test_size=0.20,
    random_state=42
)

score_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        max_depth=8,
        min_samples_leaf=1
    ))
])

score_model.fit(X_train, y_train)

print("Score reconstruction model trained successfully!")

Score reconstruction model trained successfully!


In [9]:
predicted_scores = score_model.predict(X_test)

mae = mean_absolute_error(y_test, predicted_scores)
rmse = np.sqrt(mean_squared_error(y_test, predicted_scores))
r2 = r2_score(y_test, predicted_scores)

print("SCORE RECONSTRUCTION RESULTS")
print("-----------------------------")
print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))

SCORE RECONSTRUCTION RESULTS
-----------------------------
MAE : 0.0249
RMSE: 0.0381
R²  : 0.9736


In [10]:
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X,
    y_decision,
    test_size=0.20,
    random_state=42,
    stratify=y_decision
)

decision_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        max_depth=8,
        min_samples_leaf=1
    ))
])

decision_model.fit(X_train_d, y_train_d)

predicted_decisions = decision_model.predict(X_test_d)

accuracy = accuracy_score(y_test_d, predicted_decisions)

print("DECISION RECONSTRUCTION")
print("-----------------------")
print("Accuracy:", round(accuracy, 4))
print()
print(classification_report(y_test_d, predicted_decisions))

DECISION RECONSTRUCTION
-----------------------
Accuracy: 1.0

              precision    recall  f1-score   support

     APPROVE       1.00      1.00      1.00         9
     DECLINE       1.00      1.00      1.00         1

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



In [13]:
test_input = pd.DataFrame([{
    "parameter_1": 75,
    "parameter_2": 100,
    "parameter_3": 50,
    "parameter_4": 3,
    "parameter_5": "A",
    "parameter_6": 20,
    "parameter_7": 20,
    "parameter_8": 2.5,
    "parameter_9": 300,
    "parameter_10": 0.5
}])

# Ensure the test_input columns are in the exact same order as the training features X
test_input = test_input[X.columns]

predicted_score = float(score_model.predict(test_input)[0])
predicted_decision = decision_model.predict(test_input)[0]

print("GK-03 RECONSTRUCTION TEST")
print("-------------------------")
print("Predicted score   :", round(predicted_score, 4))
print("Predicted decision:", predicted_decision)
print()
print("Observed score    : 0.8411")
print("Observed decision : APPROVE")

ValueError: X has 11 features, but RandomForestRegressor is expecting 10 features as input.

In [14]:
import joblib

joblib.dump(score_model, "GK03_score_reconstruction.pkl")
joblib.dump(decision_model, "GK03_decision_reconstruction.pkl")

print("Both reconstruction models saved successfully!")

Both reconstruction models saved successfully!
